<div class="panel panel-success">
    <div class="panel-heading">
        <b> Eklärung - Das Notebook Starten</b>
    </div>
    <div class="panel-body">

**Um das Notebook zu starten**: 
        
1. Klicke oben links in der Menüleiste auf **"Run"** und anschließend auf **"Run All Cells"**
        
**Falls ein Neustart nötig ist**:
        
1. Klicke oben links in der Menüleiste auf **"Kernel"** und anschließend auf **"Restart and Clear Outputs of All Cells"**

2. Klicke oben links in der Menüleiste auf **"Run"** und anschließend auf **"Run All Cells"**

# Daten importieren

In [24]:
from treech import ClassificationTree as ct

import pandas as pd
import numpy as np

from ipywidgets import interact, interactive, fixed, interact_manual, Layout, HBox
import ipywidgets as widgets

import plotly as py
from plotly.offline import iplot
import cufflinks as cf
cf.go_offline()

<div class="panel panel-success">
    <div class="panel-heading">
        <b> Eklärung - Daten</b>
    </div>
    <div class="panel-body">

Hier siehst du zwei Datensätze, die weiter unten zum trainieren und bewerten eines Entscheidungsbaums genutzt werden.

* **Trainingsdatensatz**: Wird genutzt, um einen Entscheidungsbaum automatisch mit einem Algorithmus zu erstellen bzw. zu trainieren.
        
    
* **Testdatensatz**: Wird genutzt, um zu bewerten wie gut ein fertiger Entscheidungsbaum funktioniert.

In [27]:
def read_csv():
    df_train = pd.read_csv('Daten/Training/jim_binaer.csv', sep = ';')
    df_test = pd.read_csv('Daten/Test/jim_binaer.csv', sep = ';')
    print('Trainingsdaten')
    display(df_train)
    print('Testdaten')
    display(df_test)
    return df_train, df_test

data_widget=interactive(read_csv)
#data_widget.result = pd.DataFrame(columns=['Importiere Datensatz'])
data_widget

interactive(children=(Output(),), _dom_classes=('widget-interact',))

# Entscheidungsbaum trainieren

<div class="panel panel-success">
    <div class="panel-heading">
        <b> Eklärung - Einen Entscheidungsbaum trainieren</b>
    </div>
    <div class="panel-body">

Mit dem folgenden Menü kannst du einen Entscheidungsbaum trainieren. D. h. ein Algorithmus wählt basierend auf den Trainingsdaten nach und nach Variablen aus und erstellt mit diesen automatisch einen Entscheidungsbaum. Du hast hier als Mensch aber noch verschiedene Möglichkeiten einzugreifen, um das Ergebnis zu beeinflussen. Du kannst die Zielvariable wählen und außerdem verschiedenen Parameter bestimmen. Diese werden beim Maschinellen Lernen auch Hyperparamter genannt.
        
Die hier aufgeführten Hyperparameter bestimmen wann der Trainingsprozess abbricht. Die 3 Parameter haben folgende Bedeutung:
        
* **Max_Tiefe**: Dieser Parameter bestimmt was die maximale Tiefe des Entscheidungsbaums sein darf. Tiefe bedeutet die Anzahl der Knoten in einem Pfad des Baums. Wenn Max_Tiefe auf 3 gesetzt wird bedeutet das also, dass immer maximal 3 Knoten in einem Pfad untereinander stehen können.

        
* **Min_Faelle**: Dieser Parameter bestimmt wie viele Fälle mindestens in einem Knoten vorhanden sein müssen, damit noch eine weitere Variable gesucht wird. Wenn Min_Faelle beispielsweise auf 75 gesetzt wird und in einem Knoten nur noch 51 Fälle vorhanden sind (z.B. eine Verteilung von [45, 6]), dann wird keine weitere Variable gesucht. Wenn hingegen 86 Fälle in einem Knoten vorhanden sind (z.B. eine Verteilung von [27, 69]), dann wird noch eine weitere Variable gesucht.


In [31]:
def grow_tree(Zielvariable,  Max_Tiefe,Min_Faelle):
    
    Krit = 'misclassification_error'
    
    tree = ct.DecisionTree(data = data_widget.result[0], target = Zielvariable, crit = Krit)
    tree.grow_tree(max_depth = Max_Tiefe-1, crit = Krit, min_leaf_cases = Min_Faelle,)
    tree.print_tree()
    display(tree.tree_graph)

    print('Rate der korrekten Klassifikationen: \n')
    print('Trainingsdaten',round((tree.prediction_accuracy(data_widget.result[0]))*100,1),'% \n')
    print('Testdaten',round(tree.prediction_accuracy(data_widget.result[1])*100,1),'%') 


    return tree
    
tree_widget = interactive(grow_tree,{'manual': True, 'manual_name': 'Erstelle Baum'}, Zielvariable = data_widget.result[0].columns.sort_values(), Max_Tiefe=(1,20,1), Min_Faelle=(1,len(data_widget.result[0]),1)) 
tree_widget


interactive(children=(Dropdown(description='Zielvariable', options=('Computer_Besitz', 'E_Reader_Besitz', 'Fes…

## Verlauf des Trainingsprozesses Visualisieren

In [34]:
def analyse_training():
    tree_widget.result.evaluation_depth(data_widget.result[0],data_widget.result[1]).iplot(xTitle='Tiefe des Baums', yTitle='Rate korrekter Klassifikationen')
   
analyse_tree_widget = interactive(analyse_training,{'manual': True, 'manual_name': 'Verlauf analysieren'})
analyse_tree_widget


interactive(children=(Button(description='Verlauf analysieren', style=ButtonStyle()), Output()), _dom_classes=…

## Entscheidungsbaum  manuell beschneiden (Pruning)

<div class="panel panel-success">
    <div class="panel-heading">
        <b> Eklärung - Einen Entscheidungsbaum beschneiden</b>
    </div>
    <div class="panel-body">

Mit dem folgenden Menu kannst du den Entscheidungsbaum beschneiden, den du weiter oben erstellt hast. Du kannst dir einen beliebigen Knoten aussuchen und den kompletten darunter liegenden Pfad abschneiden. In jedem Knoten wird unten rechts eine **Knotennummer** angezeigt. Du musst nur die passende Knotennummer hier eintragen.
        
Wenn du einen Haken bei **"permanent"** setzt wird der Knoten permanent abgeschnitten. Wenn der Haken nicht gesetzt ist, wird der Knoten nur probeweise abgeschnitten. Dieser taucht dann anschließend wieder auf, wenn du einen anderen Knoten abschneidest.

In [38]:
def prune_trial(Knoten_nr, permanent):
    
    if permanent:
        tree_widget.result.prune_node(int(Knoten_nr))
        tree_widget.result.print_tree()
        display(tree_widget.result.tree_graph)
        
        print('Rate der korrekten Klassifizierungen: \n')
        print('Trainingsdaten',round((tree_widget.result.prediction_accuracy(data_widget.result[0]))*100,5),'% \n')
        print('Testdaten',round(tree_widget.result.prediction_accuracy(data_widget.result[1])*100,5),'%')      
    else:
        current_tree = tree_widget.result.copy()
        current_tree.prune_node(int(Knoten_nr))
        current_tree.print_tree()
        display(current_tree.tree_graph)
    
        print('Rate der korrekten Klassifikationen: \n')
        print('Trainingsdaten',round((current_tree.prediction_accuracy(data_widget.result[0]))*100,5),'% \n')
        print('Testdaten',round(current_tree.prediction_accuracy(data_widget.result[1])*100,5),'%')      
    
    
        return current_tree
    
prune_widget_trial = interactive(prune_trial,{'manual': True, 'manual_name': 'Knoten abschneiden'}, Knoten_nr = '14',  permanent = False)
prune_widget_trial

interactive(children=(Text(value='14', continuous_update=False, description='Knoten_nr'), Checkbox(value=False…

## Entscheidungsbaum Ausgeben

In [41]:
def output_tree():       
    tree_widget.result.tree_graph.view()
    
output_tree_widget = interactive(output_tree,{'manual': True, 'manual_name': 'Tree ausgeben'})
output_tree_widget


interactive(children=(Button(description='Tree ausgeben', style=ButtonStyle()), Output()), _dom_classes=('widg…